In [1]:
# All installs in one cell — run once per session
# pip install -r requirements.txt

!python -m spacy download en_core_web_sm -q

print("Environment ready.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
Environment ready.


# Setup Local Environment

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Create project folder locally
PROJECT_DIR = Path("./").resolve()
CHROMA_DIR  = Path(os.environ.get("CHROMA_PERSIST_DIR", "data/chroma")).resolve()
RAW_DIR     = PROJECT_DIR / "data" / "raw_kanoon"

for d in [PROJECT_DIR, CHROMA_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")
print(f"ChromaDB dir: {CHROMA_DIR}")
print(f"Raw data dir: {RAW_DIR}")

Project dir: D:\Projects\NyayaAgent
ChromaDB dir: D:\Projects\NyayaAgent\data\chroma
Raw data dir: D:\Projects\NyayaAgent\data\raw_kanoon


# API keys

In [3]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

KANOON_KEY = os.getenv("KANOON_API_KEY")
if not KANOON_KEY:
    KANOON_KEY = getpass.getpass("Indian Kanoon API key: ")

OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_KEY:
    OPENROUTER_KEY = getpass.getpass("OpenRouter API key: ")

print("API keys loaded.")

API keys loaded.


# Fetcher

In [4]:
import requests, time, json

KANOON_API = "https://api.indiankanoon.org"
HEADERS    = {"Authorization": f"Token {KANOON_KEY}"}

def search_judgments(query: str, max_pages: int = 5) -> list[dict]:
    docs = []
    for page in range(max_pages):
        resp = requests.post(
            f"{KANOON_API}/search/",
            data={"formInput": query, "pagenum": page},
            headers=HEADERS, timeout=30,
        )
        resp.raise_for_status()
        batch = resp.json().get("docs", [])
        if not batch:
            break
        docs.extend(batch)
        time.sleep(0.6)
    return docs

def fetch_document(tid: int) -> dict | None:
    cache = RAW_DIR / f"{tid}.json"    # Drive-backed cache
    if cache.exists():
        return json.loads(cache.read_text())
    resp = requests.post(
        f"{KANOON_API}/doc/{tid}/",
        headers=HEADERS, timeout=30,
    )
    if resp.status_code != 200:
        return None
    data = resp.json()
    cache.write_text(json.dumps(data, ensure_ascii=False))
    time.sleep(0.5)
    return data

def fetch_corpus(queries: list[str], max_per_query: int = 50) -> list[dict]:
    all_docs, seen = [], set()
    for query in queries:
        stubs = search_judgments(query, max_pages=max_per_query // 10)
        for stub in stubs:
            tid = stub.get("tid")
            if not tid or tid in seen:
                continue
            doc = fetch_document(tid)
            if doc:
                all_docs.append(doc)
                seen.add(tid)
        print(f"  '{query[:40]}': {len(seen)} docs fetched")
    return all_docs

# Fetch — start small to test, then increase max_per_query
QUERIES = [
    "insider trading SEBI penalty Supreme Court",
    "contract breach specific performance",
    "fundamental rights constitutional law",
    "criminal offence battery charges",
    "property dispute specific guidelines",
    "piracy and intellectual property theft"
]

print("Fetching from Indian Kanoon...")
raw_docs = fetch_corpus(QUERIES, max_per_query=50)
print(f"\nTotal: {len(raw_docs)} documents")

Fetching from Indian Kanoon...
  'insider trading SEBI penalty Supreme Cou': 50 docs fetched
  'contract breach specific performance': 100 docs fetched
  'fundamental rights constitutional law': 150 docs fetched
  'criminal offence battery charges': 198 docs fetched
  'property dispute specific guidelines': 247 docs fetched
  'piracy and intellectual property theft': 277 docs fetched

Total: 277 documents


# Preprocessing + chunking

In [5]:
import re, unicodedata, spacy
from bs4 import BeautifulSoup
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42

nlp = spacy.load("en_core_web_sm")
nlp.max_length = 2_000_000

BOILERPLATE_RE = re.compile(
    r'Take notes as you read.*?judgment|Print this judgment|'
    r'Download this judgment|Try out our Premium.*?FREE',
    re.IGNORECASE | re.DOTALL
)
CITATION_NORMALISE = [
    (re.compile(r'\(\s*(\d{4})\s*\)'),     r'(\1)'),
    (re.compile(r'S\s*\.?\s*C\s*\.?\s*C'), r'SCC'),
    (re.compile(r'A\s*\.?\s*I\s*\.?\s*R'), r'AIR'),
]
HEADING_RE = re.compile(
    r'\n(?=(?:JUDGMENT|ORDER|HELD|FACTS|RATIO|OBITER|PER [A-Z]+|\d+\.\s+[A-Z]))',
    re.MULTILINE
)
LABELS_MAP = {
    "JUDGMENT":"judgment","ORDER":"order","HELD":"ratio",
    "FACTS":"facts","RATIO":"ratio","OBITER":"obiter","PER ":"opinion"
}

def extract_and_preprocess(raw: dict) -> dict | None:
    html  = raw.get("doc", "")
    soup  = BeautifulSoup(html, "html.parser")
    for tag in soup(["script","style","nav","button"]): tag.decompose()
    text  = soup.get_text(separator="\n")
    text  = BOILERPLATE_RE.sub("", text)
    text  = re.sub(r'\n{3,}', '\n\n', text)
    text  = re.sub(r'[ \t]{2,}', ' ', text)
    text  = re.sub(r'\u00a0', ' ', text)
    text  = unicodedata.normalize("NFKC", text).strip()
    for pat, rep in CITATION_NORMALISE: text = pat.sub(rep, text)
    if len(text) < 500: return None
    try:
        lang = detect(text[:2000])
    except Exception:
        lang = "en"
    ascii_ratio = sum(1 for c in text if ord(c) < 128) / max(len(text), 1)
    if lang not in ("en","unknown") and ascii_ratio < 0.60: return None
    return {
        "tid":   str(raw.get("tid","")),
        "title": raw.get("title","")[:300],
        "court": raw.get("court",""),
        "year":  (raw.get("publishdate","") or "")[:4],
        "text":  text, "language": lang,
    }

def chunk_document(doc: dict, max_tok=400, overlap_tok=64) -> list[dict]:
    parts = HEADING_RE.split(doc["text"])
    chunks, idx = [], 0
    for part in parts:
        label = "general"
        for kw, tag in LABELS_MAP.items():
            if part.strip().upper().startswith(kw):
                label = tag; break
        spacy_doc  = nlp(part[:500_000])
        sentences  = [s.text.strip() for s in spacy_doc.sents if s.text.strip()]
        window, wtok = [], 0
        for sent in sentences:
            stok = int(len(sent.split()) * 1.3)
            if wtok + stok > max_tok and window:
                chunks.append({**{k:v for k,v in doc.items() if k!="text"},
                    "chunk_text": " ".join(window),
                    "chunk_index": idx, "section": label})
                idx += 1
                overlap, otok = [], 0
                for s in reversed(window):
                    st = int(len(s.split())*1.3)
                    if otok+st > overlap_tok: break
                    overlap.insert(0, s); otok += st
                window, wtok = overlap, otok
            window.append(sent); wtok += stok
        if window:
            chunks.append({**{k:v for k,v in doc.items() if k!="text"},
                "chunk_text": " ".join(window),
                "chunk_index": idx, "section": label})
            idx += 1
    return chunks

from tqdm.auto import tqdm   # Colab shows progress bars inline

print("Preprocessing...")
processed = []
for raw in tqdm(raw_docs, desc="preprocess"):
    doc = extract_and_preprocess(raw)
    if doc: processed.append(doc)
print(f"{len(processed)} docs after filtering")

print("\nChunking...")
all_chunks = []
for doc in tqdm(processed, desc="chunking"):
    all_chunks.extend(chunk_document(doc))
print(f"{len(all_chunks)} chunks created")

d:\Programs\Anaconda\envs\nyaya-langgraph\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Preprocessing...


preprocess: 100%|██████████| 277/277 [00:21<00:00, 12.81it/s]


277 docs after filtering

Chunking...


chunking: 100%|██████████| 277/277 [42:13<00:00,  9.15s/it] 

48900 chunks created


# Embedding

In [6]:
from sentence_transformers import SentenceTransformer
import torch, numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Embedding on: {device}")
if device == "cpu":
    print("WARNING: CPU embedding is slow. ~40 chunks/min.")
    print("Tip: Runtime → Change runtime type → T4 GPU")

print("Loading InLegalBERT (downloads ~440 MB on first run)...")
model = SentenceTransformer("law-ai/InLegalBERT", device=device)
print("Model loaded.")

BATCH = 64 if device == "cuda" else 16

texts = [c["chunk_text"] for c in all_chunks]
print(f"\nEmbedding {len(texts)} chunks (batch={BATCH})...")

vectors = model.encode(
    texts,
    batch_size=BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,   # L2 normalise for cosine
    convert_to_numpy=True,
)
print(f"Shape: {vectors.shape}")    # (N, 768)

# Attach embeddings to chunks
for chunk, vec in zip(all_chunks, vectors):
    chunk["embedding"] = vec.tolist()

print("Embeddings attached to chunks.")

Embedding on: cuda
Loading InLegalBERT (downloads ~440 MB on first run)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30287.63it/s]
[transformers] BertModel LOAD REPORT from: law-ai/InLegalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.

Embedding 48900 chunks (batch=64)...


Batches: 100%|██████████| 765/765 [20:27<00:00,  1.60s/it]


Shape: (48900, 768)
Embeddings attached to chunks.


# ChromaDB storage

In [7]:
import chromadb
from chromadb.config import Settings

# Point ChromaDB at Google Drive — survives session resets
client = chromadb.PersistentClient(
    path=str(CHROMA_DIR),
    settings=Settings(anonymized_telemetry=False),
)

collection = client.get_or_create_collection(
    name="nyaya_kanoon",
    metadata={
        "hnsw:space":           "cosine",
        "hnsw:construction_ef": 200,
        "hnsw:M":               64,
    }
)
print(f"Collection ready — {collection.count()} existing docs")

# Upsert in batches of 500
BATCH = 500
for i in tqdm(range(0, len(all_chunks), BATCH), desc="storing"):
    batch = all_chunks[i : i+BATCH]
    collection.upsert(
        ids        = [f"{c['tid']}_{c['chunk_index']}" for c in batch],
        documents  = [c["chunk_text"]  for c in batch],
        embeddings = [c["embedding"]   for c in batch],
        metadatas  = [{
            "tid":     c.get("tid",""),
            "title":   c.get("title","")[:200],
            "court":   c.get("court",""),
            "year":    c.get("year",""),
            "section": c.get("section","general"),
            "language":c.get("language","en"),
        } for c in batch],
    )

print(f"\nDone. {collection.count()} total chunks stored.")

Collection ready — 0 existing docs


storing: 100%|██████████| 98/98 [00:56<00:00,  1.73it/s]


Done. 48900 total chunks stored.


# RAG query with OpenRouter Gemma

In [1]:
import requests as req

OPENROUTER_API = "https://openrouter.ai/api/v1/chat/completions"
GEMMA_MODEL    = "inclusionai/ling-2.6-1t:free"

SYSTEM_PROMPT = """You are Nyaya Agent, a senior Indian legal research assistant.
Answer using ONLY the retrieved context. Cite every claim as [N].
If context is insufficient, say so — never invent citations."""

def rag_query(question: str, n_results: int = 6, year_from: str = None) -> str:
    # 1. Embed query
    q_vec = model.encode(
        [question], normalize_embeddings=True
    )[0].tolist()

    # 2. Retrieve from ChromaDB
    where = {"year": {"$gte": int(year_from)}} if year_from else None # Convert year_from to int
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=n_results,
        where=where,
        include=["documents","metadatas","distances"],
    )

    # 3. Build context block
    context_lines = []
    for i, (doc, meta, dist) in enumerate(zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ), 1):
        sim = round(1 - dist, 3)
        context_lines.append(
            (
                f"[{i}] {meta.get('title','?')[:80]} | "
                f"{meta.get('court','')} | {meta.get('year','')} "
                f"| section: {meta.get('section','')} | sim: {sim}\n"
                f"{doc[:600]}"
            )
        )
    context = "\n\n---\n\n".join(context_lines)

    # 4. Call Gemma via OpenRouter
    resp = req.post(
        OPENROUTER_API,
        json={
            "model": GEMMA_MODEL,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content":
                    f"Context:\n\n{context}\n\nQuestion: {question}"},
            ],
            "max_tokens": 1200,
            "temperature": 0.1,
        },
        headers={
            "Authorization": f"Bearer {OPENROUTER_KEY}",
            "Content-Type":  "application/json",
            "HTTP-Referer":  "https://nyaya-agent.local",
            "X-Title":       "Nyaya Agent",
        },
        timeout=60,
    )
    print(resp)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

# Test it
answer = rag_query(
    "What is the penalty for insider trading under SEBI regulations?",
    n_results=6,
    year_from="2015",
)
print(answer)

NameError: name 'model' is not defined